In [1]:
import numpy as np
import soundfile as sf
from datasets import load_dataset, Audio
from transformers import EncodecModel, AutoProcessor, EncodecFeatureExtractor

# dummy dataset, however you can swap this with an dataset on the 🤗 hub or bring your own

# load the model + processor (for pre-processing the audio)
model = EncodecModel.from_pretrained("facebook/encodec_48khz")
processor = AutoProcessor.from_pretrained("facebook/encodec_48khz")


/home/sgarc/LowGen/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration.


In [2]:
stereo = sf.read("./stereo.wav")

In [3]:
stereo[0][0]

array([-0.00104201, -0.00073826])

In [4]:
stereo_data = np.transpose(stereo[0])

In [5]:
inputs = processor(raw_audio=stereo_data, sampling_rate=processor.sampling_rate, return_tensors="pt")

In [6]:
encoder_outputs = model.encode(inputs["input_values"], inputs["padding_mask"])

In [7]:
encoder_outputs.audio_codes.shape

torch.Size([264, 1, 2, 150])

In [8]:
audio_values = model.decode(encoder_outputs.audio_codes, encoder_outputs.audio_scales, inputs["padding_mask"])

: 

In [ ]:
audio_values.audio_values[0][0]

tensor([0.0007, 0.0002, 0.0003,  ..., 0.0001, 0.0001, 0.0001],
       grad_fn=<SelectBackward0>)

In [ ]:
# save the audio sample to disk
sf.write("audio_sample_reconstructed.wav", audio_values.audio_values[0][0].tolist(), processor.sampling_rate)